# Stage 2 - ExtraSensory mechanism comparison (Colab runner)

Runs the repository's own Stage 2 code (`experiment_Files/Stage2/`). Nothing is re-implemented here.

**Fixed in this version**
* Cell 4 previously used a shell `for` loop inside an IPython `!` line, where `$f` is substituted from the
  *Python* namespace. `f` did not exist there, so nothing was decompressed and the CSV count stayed 0.
  Data preparation is now pure Python (`scripts/prepare_data.py`), with md5 verification and a progress bar.
* Cell 2 previously force-downgraded numpy/pandas/scikit-learn, which is slow and can demand a runtime
  restart. It now uses Colab's preinstalled versions, installs only what is missing, and records the actual
  versions in the run metadata.
* This notebook contains **no shell magics at all**, so there are no further `$variable` substitution traps.

**Order:** run cells 1 to 9 top to bottom. **A CPU runtime is fine** - scikit-learn does not use the GPU.

**If the session drops:** rerun cells 1-3, then cell 6. Every runner supports `--resume` and continues from
the users already completed.

In [ ]:
#@title 1. Configuration
REPO_URL    = "https://github.com/TerryBinful/contectAware.git"  #@param {type:"string"}
BRANCH      = "main"  #@param {type:"string"}
TASK        = "v2_offline_analysis"  #@param ["v2_offline_analysis", "v2_scoredump_f3", "v2_scoredump_f7", "mechanism_comparison", "sensitivity_far_003", "sensitivity_far_007", "stage2_ablation", "dataset_audit_only"]
MAX_USERS   = 0  #@param {type:"integer"}
PUSH_RESULTS = True  #@param {type:"boolean"}
TOKEN_SECRET = "GITHUB_TOKEN"  #@param {type:"string"}
CACHE_ON_DRIVE = True  #@param {type:"boolean"}
GIT_NAME    = "Stage2 Colab Runner"  #@param {type:"string"}
GIT_EMAIL   = "colab@example.com"  #@param {type:"string"}

REPO_DIR  = "/content/repo"
STAGE2    = REPO_DIR + "/experiment_Files/Stage2"
DRIVE_DIR = "/content/drive/MyDrive/extrasensory_cache"
DATA_DIR  = DRIVE_DIR + "/csv" if CACHE_ON_DRIVE else "/content/extrasensory_csv"
CONFIG = OUT = None
print("task:", TASK, "| users:", MAX_USERS or "all", "| push:", PUSH_RESULTS, "| data:", DATA_DIR)

In [ ]:
#@title 2. Environment (uses Colab's preinstalled stack; installs only what is missing)
import importlib, subprocess, sys
need = []
for mod, pkg in [("numpy", "numpy"), ("pandas", "pandas"), ("sklearn", "scikit-learn"),
                 ("scipy", "scipy"), ("matplotlib", "matplotlib"), ("tabulate", "tabulate")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        need.append(pkg)
if need:
    print("installing:", need)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *need], check=True)
else:
    print("all required packages already present - nothing to install")

import numpy, pandas, sklearn, scipy
VERSIONS = dict(python=sys.version.split()[0], numpy=numpy.__version__, pandas=pandas.__version__,
                sklearn=sklearn.__version__, scipy=scipy.__version__)
print(VERSIONS)
print("Versions are recorded in each run's experiment_metadata.json. They may differ from the Stage 1")
print("pins (numpy 2.0.2 / pandas 2.2.2 / sklearn 1.6.1): results are seed-deterministic but not")
print("guaranteed bit-identical across library versions.")

In [ ]:
#@title 3. Repository (clone; authenticated if a token secret is available)
import os, shutil, subprocess
TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get(TOKEN_SECRET)
    print("token secret found:", TOKEN_SECRET)
except Exception as e:
    print("no token secret (", type(e).__name__, ") - results can still be downloaded as a zip in cell 9")

clone_url = REPO_URL if not TOKEN else REPO_URL.replace("https://", "https://" + TOKEN + "@")
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
r = subprocess.run(["git", "clone", "-q", "--branch", BRANCH, clone_url, REPO_DIR],
                   capture_output=True, text=True)
err = (r.stderr or "").replace(TOKEN, "***") if TOKEN else (r.stderr or "")
assert r.returncode == 0, "clone failed: " + err
subprocess.run(["git", "config", "user.name", GIT_NAME], cwd=REPO_DIR)
subprocess.run(["git", "config", "user.email", GIT_EMAIL], cwd=REPO_DIR)
assert os.path.isdir(STAGE2), "Stage 2 package not found at " + STAGE2
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR,
                        capture_output=True, text=True).stdout.strip()
print("cloned", BRANCH, "at", COMMIT)
print(sorted(os.listdir(STAGE2)))

In [ ]:
#@title 4. Data (pure Python: download, md5-verify, decompress; cached on Drive if enabled)
import os, sys
if TASK == "v2_offline_analysis":
    print("v2_offline_analysis reads only the committed score dumps - no dataset needed. Skipping.")
if CACHE_ON_DRIVE and TASK != "v2_offline_analysis":
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
if TASK != "v2_offline_analysis":
    sys.path.insert(0, STAGE2 + "/scripts")
    import prepare_data
    prepare_data.prepare(DATA_DIR, work_dir=DRIVE_DIR if CACHE_ON_DRIVE else "/content")

In [ ]:
#@title 5. Pre-flight audit (schema, participants, planned runs; trains nothing)
import subprocess, sys
CONFIGS = {"v2_scoredump_f3":  ("configs/v2_scoredump_f3.json",  "results/mechanism_comparison_v2/f3"),
           "v2_scoredump_f7":  ("configs/v2_scoredump_f7.json",  "results/mechanism_comparison_v2/f7"),
           "mechanism_comparison": ("configs/mechanism_comparison.json", "results/mechanism_comparison"),
           "sensitivity_far_003": ("configs/sensitivity_far_003.json", "results/mechanism_comparison/sensitivity/far_003"),
           "sensitivity_far_007": ("configs/sensitivity_far_007.json", "results/mechanism_comparison/sensitivity/far_007"),
           "stage2_ablation":     ("configs/stage2_primary.json", "results/primary")}
if TASK == "v2_offline_analysis":
    # Frozen dumps in, decision layer out. Fits nothing; ~70 s for 31 users.
    OUT = "results/mechanism_comparison_v2/analysis"
    cmd = [sys.executable, "scripts/decision_layer_offline.py",
           "--config", "configs/v2_scoredump_f3.json",
           "--scores", "results/mechanism_comparison_v2/f3/scores",
           "--f7-scores", "results/mechanism_comparison_v2/f7/scores",
           "--out", OUT]
    p = subprocess.run(cmd, cwd=STAGE2, capture_output=True, text=True)
    print(p.stdout[-5000:]); print(p.stderr[-2000:])
    assert p.returncode == 0, "offline analysis failed"
elif TASK == "dataset_audit_only":
    cmd = [sys.executable, "scripts/run_dataset_audit.py", "--data", DATA_DIR, "--out", "results/dataset_audit"]
else:
    CONFIG, OUT = CONFIGS[TASK]
    driver = "scripts/run_stage2.py" if TASK == "stage2_ablation" else "scripts/run_mechanism_comparison.py"
    cmd = [sys.executable, driver, "--config", CONFIG, "--data", DATA_DIR, "--out", OUT, "--dry-run"]
p = subprocess.run(cmd, cwd=STAGE2, capture_output=True, text=True)
print(p.stdout[-4000:])
print(p.stderr[-2000:])
assert p.returncode == 0, "pre-flight failed - read the message above before continuing"
print("CONFIG:", CONFIG, "| OUT:", OUT)

In [ ]:
#@title 6. Run (long; the log streams live; safe to rerun - it resumes)
import subprocess, sys, time
if TASK in ("dataset_audit_only", "v2_offline_analysis"):
    print("cell 5 already completed this task; nothing further to run.")
else:
    driver = "scripts/run_stage2.py" if TASK == "stage2_ablation" else "scripts/run_mechanism_comparison.py"
    cmd = [sys.executable, "-u", driver, "--config", CONFIG, "--data", DATA_DIR, "--out", OUT, "--resume"]
    if MAX_USERS:
        cmd += ["--max-users", str(MAX_USERS)]
    print(" ".join(cmd))
    t0 = time.time()
    pr = subprocess.Popen(cmd, cwd=STAGE2, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1)
    for line in pr.stdout:
        print(line, end="")
    pr.wait()
    print("return code", pr.returncode, "| wall time %.1f min" % ((time.time() - t0) / 60))
    assert pr.returncode == 0, "run failed - rerun this cell to resume from the last completed user" 

In [ ]:
#@title 7. Analysis, figures and tables
import subprocess, sys
if TASK == "v2_offline_analysis":
    print("analysis already written by cell 5; run cell 8 (tests) then cell 9 (push).")
elif TASK.startswith("v2_"):
    print("v2 score-dump run: the decision layer is applied OFFLINE by")
    print("scripts/decision_layer_offline.py per docs/Stage2/PREREGISTRATION_v2.md.")
    print("Nothing to analyse in this notebook - return the zip from cell 9.")
elif TASK in ("mechanism_comparison", "sensitivity_far_003", "sensitivity_far_007"):
    p = subprocess.run([sys.executable, "scripts/analyse_mechanism_comparison.py", "--out", OUT],
                       cwd=STAGE2, capture_output=True, text=True)
    print(p.stdout[-6000:])
    print(p.stderr[-2000:])
elif TASK == "stage2_ablation":
    p = subprocess.run([sys.executable, "scripts/make_report.py", "--out", OUT,
                        "--report", "../../docs/Stage2/STAGE2_EXECUTION_REPORT.md"],
                       cwd=STAGE2, capture_output=True, text=True)
    print(p.stdout[-4000:])
else:
    print("no analysis step for this task")

In [ ]:
#@title 8. Tests (unit + protocol/leakage) - results are not trustworthy unless these pass
import subprocess, sys, os
target = "results/mechanism_comparison" if (TASK == "dataset_audit_only" or TASK.startswith("v2_")) else OUT
cmds = [[sys.executable, "tests/test_mechanisms.py"], [sys.executable, "tests/test_decision_v2.py"]]
if os.path.exists(os.path.join(STAGE2, target, "sequence_metrics.csv")):
    cmds.append([sys.executable, "tests/test_leakage.py", target])
for cmd in cmds:
    p = subprocess.run(cmd, cwd=STAGE2, capture_output=True, text=True)
    print((p.stdout or p.stderr)[-2500:])
    print("-> return code", p.returncode)

In [ ]:
#@title 9. Write results back to the repository (commit + push), or download a zip
import subprocess, os, shutil
def git(*args):
    return subprocess.run(["git", *args], cwd=REPO_DIR, capture_output=True, text=True)

status = git("status", "--short").stdout
print(status[:3000] or "(no changes)")

if PUSH_RESULTS and TOKEN and status.strip():
    git("add", "-A", "experiment_Files/Stage2", "docs/Stage2")
    msg = "colab: " + TASK + " results (from " + COMMIT + ", sklearn " + VERSIONS["sklearn"] + ")"
    print(git("commit", "-m", msg).stdout[-1500:])
    pull = git("pull", "--rebase", "origin", BRANCH)
    print(pull.stdout[-600:], pull.stderr[-600:])
    push = git("push", "origin", BRANCH)
    print((push.stderr or "").replace(TOKEN, "***")[-600:])
    print("PUSHED" if push.returncode == 0 else "PUSH FAILED - use the zip below")
elif not TOKEN:
    print("no token: skipping push")

zip_base = "/content/stage2_" + TASK + "_results"
src_dir = os.path.join(STAGE2, "results", "mechanism_comparison_v2") if TASK.startswith("v2_") else os.path.join(STAGE2, "results")
shutil.make_archive(zip_base, "zip", src_dir)
print(zip_base + ".zip", "(%.1f MB)" % (os.path.getsize(zip_base + ".zip") / 1e6))
try:
    from google.colab import files
    files.download(zip_base + ".zip")
except Exception as e:
    print("download from the Files pane:", e)